# Ingesta Full e Incremental — Superstore (Microsoft Fabric)

**Autor:** Eduardo Osorio Venegas
**Dataset real:** `SuperStore_Tablon.xlsx` (9.800 filas, pedidos 2015–2018)

Ejemplo simplificado de las dos estrategias de carga, orquestadas con **Pipelines de Fabric** que disparan **Notebooks** (PySpark + Spark SQL).

**Arquitectura:**

```
Archivo origen (Files)
        │
        ▼
Pipeline: Ingesta_Superstore  →  Copy Data hacia Files
        │
   ┌────┴─────┐
   ▼          ▼
Notebook   Notebook
Full Load  Incremental (merge + watermark)
   └────┬─────┘
        ▼
Tabla Silver (Lakehouse) · superstore
```

Dos pipelines separados, cada uno con **una sola actividad Notebook**:

| Pipeline | Actividad | Notebook |
|---|---|---|
| `PL_Full_Load_Superstore` | Notebook (ejecución manual / única vez) | `NB_Full_Load_Superstore` |
| `PL_Incremental_Load_Superstore` | Notebook (parámetro `fecha_fin`, trigger programado) | `NB_Incremental_Load_Superstore` |

Para que el ejemplo sea manejable, vamos a **simular una línea de tiempo real** con el propio dataset:

- **Bronze** = todo el archivo tal cual llega (9.800 filas) → representa el "sistema origen".
- **Full Load** = carga inicial de todos los pedidos **anteriores a 2018-01-01** (6.542 filas) → la foto histórica.
- **Incremental** = simula la llegada de pedidos **nuevos** de 2018, en dos tandas (como si el pipeline corriera cada 6 meses):
  - Tanda 1: pedidos hasta el 30-jun-2018 (1.159 filas nuevas)
  - Tanda 2: pedidos hasta el 31-dic-2018 (2.099 filas nuevas)

No hay una columna de "última modificación" en este dataset, así que usamos `Order_Date` como watermark — es la simplificación intencional de este ejemplo.


## 0. Parámetros del notebook

En Fabric, la celda marcada con el tag **`parameters`** es la que el Pipeline sobrescribe al invocar el notebook desde una actividad **Notebook** (`Base parameters`). Acá dejamos un valor por defecto para poder probar el notebook manualmente.


In [10]:
# Parámetro que el Pipeline inyecta en cada ejecución incremental.
# Simula "hasta qué fecha llegan los pedidos en este batch".
fecha_fin = "2018-06-30"

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 12, Finished, Available, Finished, False)

## 1. Landing: leer el archivo origen hacia Bronze

El archivo `.xlsx` llega a `Files/raw/superstore/` vía una actividad **Copy Data** del pipeline. Spark no lee `.xlsx` nativamente, así que lo abrimos con `pandas` y lo convertimos a Spark DataFrame — un patrón muy habitual en Fabric para archivos Excel pequeños/medianos.


In [11]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

ruta_archivo = "/lakehouse/default/Files/EDUARDO OSORIO/XLSX/SuperStore_Tablon.xlsx"

pdf = pd.read_excel(ruta_archivo)
pdf["Order_Date"] = pd.to_datetime(pdf["Order_Date"]).dt.date
pdf["Ship_Date"] = pd.to_datetime(pdf["Ship_Date"]).dt.date

df_bronze = spark.createDataFrame(pdf)

print(f"Filas leídas del origen: {df_bronze.count()}")
df_bronze.printSchema()


StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 13, Finished, Available, Finished, False)

Filas leídas del origen: 9800
root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)



In [12]:
# Guardamos Bronze como tabla administrada Delta (representa el "origen" ya aterrizado)
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("LH_BRONZE_ESSENTIALS.dbo.superstore_full")
)

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 14, Finished, Available, Finished, False)

In [13]:
%%sql
SELECT
    COUNT(*)              AS total_filas,
    MIN(Order_Date)       AS fecha_minima,
    MAX(Order_Date)       AS fecha_maxima,
    COUNT(DISTINCT Row_ID) AS row_ids_unicos
FROM LH_BRONZE_ESSENTIALS.dbo.superstore_full

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

## 2. Notebook: `NB_Full_Load_Superstore`

Se ejecuta **una sola vez** (o cuando se necesita reconstruir todo el histórico). Toma los pedidos anteriores a 2018 desde Bronze y **sobrescribe por completo** la tabla Silver — sin lógica de merge, sin watermark.


In [14]:
FECHA_CORTE_HISTORICO = "2018-01-01"

df_full = spark.table("LH_BRONZE_ESSENTIALS.dbo.superstore_full").filter(F.col("Order_Date") < FECHA_CORTE_HISTORICO)

(
    df_full.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("LH_SILVER_ESSENTIALS.dbo.silver_superstore")
)

print(f"Full Load completado. Filas cargadas: {df_full.count()}")


StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 16, Finished, Available, Finished, False)

Full Load completado. Filas cargadas: 6542


In [15]:
%%sql
SELECT
    COUNT(*)        AS total_filas,
    MIN(Order_Date) AS fecha_minima,
    MAX(Order_Date) AS fecha_maxima
FROM LH_SILVER_ESSENTIALS.dbo.silver_superstore

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

## 3. Notebook: `NB_Incremental_Load_Superstore`

Se ejecuta **periódicamente** desde el pipeline `PL_Incremental_Load_Superstore`, que le pasa el parámetro `fecha_fin` (representando "hasta qué fecha llegaron pedidos en este batch").

**Lógica:**
1. Calcular el watermark actual: `MAX(Order_Date)` de la tabla Silver — no hace falta una tabla de control aparte, el propio destino nos dice hasta dónde llegamos.
2. Filtrar Bronze: `Order_Date > watermark AND Order_Date <= fecha_fin`.
3. **Merge (upsert)** contra Silver usando `Row_ID` como clave — así el notebook es **idempotente**: si se re-ejecuta con el mismo `fecha_fin`, no duplica filas.


In [18]:
from delta.tables import DeltaTable

# 1) Watermark actual = último Order_Date que ya está en Silver
watermark_actual = spark.sql("SELECT MAX(Order_Date) AS wm FROM LH_SILVER_ESSENTIALS.dbo.silver_superstore").collect()[0]["wm"]
print(f"Watermark actual en Silver: {watermark_actual}")
print(f"Cargando pedidos nuevos hasta: {fecha_fin}")

# 2) Filtrar el incremental en Bronze (el "origen")
df_incremental = (
    spark.table("LH_BRONZE_ESSENTIALS.dbo.superstore_full")
    .filter(F.col("Order_Date") > F.lit(watermark_actual))
    .filter(F.col("Order_Date") <= F.lit(fecha_fin))
)

filas_nuevas = df_incremental.count()
print(f"Filas nuevas detectadas: {filas_nuevas}")

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 20, Finished, Available, Finished, False)

Watermark actual en Silver: 2017-12-31
Cargando pedidos nuevos hasta: 2018-06-30
Filas nuevas detectadas: 1159


In [20]:
# 3) Merge / Upsert contra Silver usando Row_ID como clave de negocio
if filas_nuevas > 0:
    tabla_silver = DeltaTable.forName(spark, "LH_SILVER_ESSENTIALS.dbo.silver_superstore")

    (
        tabla_silver.alias("destino")
        .merge(
            df_incremental.alias("origen"),
            "destino.Row_ID = origen.Row_ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merge/Upsert completado sobre silver_superstore")
else:
    print("Sin novedades para este rango de fechas.")

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 22, Finished, Available, Finished, False)

Merge/Upsert completado sobre silver_superstore


In [21]:
%%sql
SELECT
    COUNT(*)        AS total_filas,
    MIN(Order_Date) AS fecha_minima,
    MAX(Order_Date) AS fecha_maxima
FROM LH_SILVER_ESSENTIALS.dbo.silver_superstore

StatementMeta(, 1f08bedd-2e01-4f2a-933d-daf2b0cf4bea, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### 3.1 Probar la segunda tanda incremental

Para ver el patrón repetirse (y comprobar que es idempotente), cambiá el parámetro y volvé a correr las celdas de la sección 3 — así es exactamente como lo haría el Pipeline en su segunda ejecución programada:

```python
fecha_fin = "2018-12-31"
```

Al re-ejecutar, el watermark ya habrá avanzado a `2018-06-30` (resultado de la tanda anterior), así que el filtro solo trae los pedidos del segundo semestre — sin volver a tocar lo ya cargado.


## 4. Orquestación con Pipelines de Fabric

Este notebook está pensado para que lo invoquen **dos pipelines** distintos (ver diagrama al inicio):

**`PL_Full_Load_Superstore`**
1. Actividad **Copy Data**: trae `SuperStore_Tablon.xlsx` desde el origen hacia `Files/raw/superstore/`.
2. Actividad **Notebook**: ejecuta `NB_Full_Load_Superstore` (sin parámetros).
3. Trigger: **manual**, o una sola vez al inicializar el proyecto.

**`PL_Incremental_Load_Superstore`**
1. Actividad **Copy Data**: actualiza el archivo/carpeta en `Files/raw/superstore/` con los datos más recientes del origen.
2. Actividad **Notebook**: ejecuta `NB_Incremental_Load_Superstore`, pasando como *Base parameter* `fecha_fin` (en un escenario real, algo como `@formatDateTime(utcnow(), 'yyyy-MM-dd')`, la fecha del propio momento de ejecución).
3. Trigger: **programado** (Schedule) — por ejemplo, diario o mensual según qué tan seguido llegan pedidos nuevos.

Como el notebook obtiene el watermark de la propia tabla Silver (`MAX(Order_Date)`), el pipeline incremental **no necesita** una tabla de control adicional ni un Lookup previo — lo cual simplifica bastante el diseño frente a escenarios con múltiples orígenes o con necesidad de manejar borrados.


## 5. Resumen

| | Full Load | Incremental Load |
|---|---|---|
| Notebook | `NB_Full_Load_Superstore` | `NB_Incremental_Load_Superstore` |
| Filtro | `Order_Date < '2018-01-01'` | `Order_Date > watermark AND Order_Date <= fecha_fin` |
| Escritura | `overwrite` | `MERGE` (upsert por `Row_ID`) |
| Watermark | No aplica | `MAX(Order_Date)` de la propia tabla Silver |
| Frecuencia | Una vez / bajo demanda | Programada (trigger del pipeline) |
| Filas en este ejemplo | 6.542 | 1.159 + 2.099 (dos tandas) |

Esta es la versión "sencilla" del patrón: sin tabla de control separada, sin CDC y sin manejo explícito de borrados. Si más adelante el dataset tuviera updates reales o necesitaras detectar eliminaciones, se le suman las técnicas de la sección de CDC / borrados del notebook anterior.
